In [47]:
import celldega as dega
from glob import glob
import pandas as pd
import tifffile

/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 1.14.5 when it was built against 1.14.6, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [43]:
dataset_name = 'E12_71'
path_data = 'data/IST_data/Substrate_' + dataset_name + '/'
path_landscape_files = 'data/IST_landscape_files/'

In [38]:
inst_slice = 'T8'
image_scale = 1.0
suffix = '.webp[Q=100]'

## Image

In [45]:
# Path to your OME-TIFF file
file_path = path_data + 'registered_images/' + inst_slice + '_' + dataset_name + '.ome.tiff'

# Open the OME-TIFF file and read the image data
with tifffile.TiffFile(file_path) as tif:
    series = tif.series[0] 
    image_data = series.asarray()

In [49]:
# image_data_scaled = image_data[:,:0] * 2
# Save the image data to a regular TIFF file without compression
tifffile.imwrite(path_landscape_files + 'output_regular.tif', image_data, compression=None)
# image_ds = dega.pre.reduce_image_size(path_landscape_files + 'output_regular.tif', image_scale, path_landscape_files)
image_png = dega.pre._convert_to_png(path_landscape_files + 'output_regular.tif')
dega.pre.make_deepzoom_pyramid(image_png, path_landscape_files + 'pyramid_images/', 'h&e', suffix=suffix)

# Spots

In [16]:
tsv_file = path_data + 'Substrate_E14_62_map_file.tsv' 

'data/IST_data/Substrate_E12_71/Substrate_E14_62_map_file.tsv'

In [29]:
# Define parameters
tsv_file = path_data + 'Substrate_E12_71_map_file.tsv' 
chunk_size = 10_000_000
parquet_prefix = path_landscape_files + 'map_parquet_files/output_chunk'

for i, chunk in enumerate(pd.read_csv(tsv_file, sep="\t", chunksize=chunk_size, header=None, index_col=0)):
    output_file = f"{parquet_prefix}_{i}.parquet"
    chunk.index.name = None
    chunk.to_parquet(output_file, engine="pyarrow")

    if i%20 == 0:
        print(f"Saved {output_file}")

print("Processing complete!")

# Region Barcodes

In [31]:
barcodes = pd.read_csv(
    path_data + 'matrix_files/T1_E12_71/T1_E12_71_raw/barcodes.tsv.gz', 
    sep='\t', 
    header=None, 
    index_col=0
)
barcodes.index.name = None
barcodes['x'] = pd.Series(index=barcodes.index.tolist())
barcodes['y'] = pd.Series(index=barcodes.index.tolist())

In [34]:
barcodes_list = barcodes.index.tolist()

for inst_file in glob(path_landscape_files + 'map_parquet_files/*.parquet'):
    
    inst_chunk = pd.read_parquet(inst_file)

    common_barcodes = list(set(inst_chunk.index.tolist()).intersection(barcodes_list))

    print(inst_file, 'found', len(common_barcodes), 'barcodes')

    if len(common_barcodes) > 0:
        barcodes.loc[common_barcodes, 'x'] = inst_chunk.loc[common_barcodes, 1]
        barcodes.loc[common_barcodes, 'y'] = inst_chunk.loc[common_barcodes, 2]
        

data/IST_landscape_files/map_parquet_files/output_chunk_50.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_40.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_2.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_32.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_22.parquet found 2344732 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_49.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_59.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_14.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_58.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_48.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_15.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_41.parquet foun

In [35]:
barcodes.to_parquet(path_landscape_files + 'meta_spots.parquet')

## Cells

In [50]:
cells = pd.read_csv(
    path_data + '/matrix_files/' + inst_slice + '_' + dataset_name + '/' + inst_slice + '_' + dataset_name + '_cell_binned/barcodes.tsv.gz', 
    sep='\t', 
    header=None, 
    index_col=0
)

In [52]:
cells.head()

""
0
cell100000:11721:24048
cell100001:11837:24263
cell100002:12260:23447
cell100003:11841:24234
cell100004:12301:23881


In [53]:
high_res_scale = 1/0.382

In [57]:
gc = pd.read_csv(path_data + 'registered_images/globalpos_' + dataset_name + '.csv', index_col=0)
gc

,X_shift,Y_shift
Sample_ID,,
T1_E12_71,4156,6187
T2_E12_71,3665,12827
T3_E12_71,2718,20134
T4_E12_71,1959,29518
T5_E12_71,2109,37553
T6_E12_71,9420,4390
T7_E12_71,9045,11593
T8_E12_71,8631,20424
T9_E12_71,8188,28995


In [61]:
tmp = pd.DataFrame([x.split(':') for x in cells.index.tolist()])
tmp.set_index(0, inplace=True)
tmp.index.name = None
tmp.columns = ['x', 'y']
tmp = tmp.astype(float)
tmp['x'] = (tmp['x'] - gc.loc[inst_slice + '_' + dataset_name, 'X_shift']) * high_res_scale 
tmp['y'] = (tmp['y'] - gc.loc[inst_slice + '_' + dataset_name, 'Y_shift']) * high_res_scale

tmp["geometry"] = tmp.apply(
    # swapped for some reason
        lambda row: [row["y"], row["x"]] , axis=1
    )

tmp['name'] = pd.Series(tmp.index.tolist(), index=tmp.index.tolist())

tmp[['name', 'geometry']].to_parquet(path_landscape_files + 'cell_metadata.parquet')

In [62]:
print(tmp.x.min(), tmp.x.max())
print(tmp.y.min(), tmp.y.max())

2251.30890052356 11507.85340314136
3678.0104712041884 14513.0890052356


In [63]:
clusters = pd.DataFrame(index=tmp.index.tolist())
clusters['cluster'] = pd.Series(0, index=tmp.index.tolist())

In [64]:

clusters.to_parquet(path_landscape_files + 'cell_clusters/cluster.parquet')

OSError: Cannot save file into a non-existent directory: 'data/IST_landscape_files/cell_clusters'